# get-children-callable-param — faded example 3: Fill __call__ delegation to forward

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `get-children-callable-param`. Running the beacon reports progress on the `Backprop: get_children callable param` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: get_children callable param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`get-children-callable-param`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "get-children-callable-param"
DD_SUBTOPIC = "Backprop: get_children callable param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The base `Module.__call__` delegates to `self.forward(*args, **kwargs)`, which is why `model(x)` works. The base `forward` raises `NotImplementedError` naming the concrete subclass.

## Faded exercise 3

Implement the base `Module` with `__call__` and `forward`. `__call__` must pass all args through to `self.forward`; the base `forward` must raise `NotImplementedError` naming the subclass. Complete the blanked `__call__` body.

**Fill in:** the __call__ body that delegates to self.forward with all args and returns the result

In [ ]:
class Module:
    def __call__(self, *args, **kwargs):
        raise NotImplementedError()  # TODO: the __call__ body that delegates to self.forward with all args and returns the result

    def forward(self, *args, **kwargs):
        raise NotImplementedError(f'{type(self).__name__} must implement forward')

class Adder(Module):
    def forward(self, a, b):
        return a + b

print(Adder()(2, 3))


def _test():
    # subclass with forward is callable and returns forward's result
    assert Adder()(2, 3) == 5
    assert Adder()(a=10, b=4) == 14

    # subclass without forward raises NotImplementedError naming itself
    class Forgot(Module):
        pass
    raised = False
    try:
        Forgot()(1)
    except NotImplementedError as e:
        raised = True
        assert 'Forgot' in str(e), str(e)
    assert raised, 'expected NotImplementedError'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class Module:
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def forward(self, *args, **kwargs):
        raise NotImplementedError(f'{type(self).__name__} must implement forward')

class Adder(Module):
    def forward(self, a, b):
        return a + b

print(Adder()(2, 3))
```
</details>